

Python virtual environments guide · MD
# Python Virtual Environments: Complete Development Guide 
 
 
## Table of Contents
 
1. [Virtual environment kya hai aur kyun chahiye?](#1-virtual-environment-kya-hai-aur-kyun-chahiye)
2. [Andar se kaise kaam karta hai](#2-andar-se-kaise-kaam-karta-hai)
3. [Pehla venv banao (step by step)](#3-pehla-venv-banao-step-by-step)
4. [pip aur requirements.txt](#4-pip-aur-requirementstxt)
5. [Git ke saath: .gitignore aur sharing](#5-git-ke-saath-gitignore-aur-sharing)
6. [VS Code mein venv select karna](#6-vs-code-mein-venv-select-karna)
7. [Real project workflows (Django / FastAPI / Flask)](#7-real-project-workflows)
8. [Python version management](#8-python-version-management)
9. [Alternatives: virtualenv, pipenv, poetry, uv, conda](#9-alternatives-tools-comparison)
10. [Docker aur venv](#10-docker-aur-venv)
11. [Common errors aur fixes](#11-common-errors-aur-fixes)
12. [Best practices](#12-best-practices)
13. [Cheat sheet](#13-cheat-sheet)
14. [Practice tasks](#14-practice-tasks)
---
 
## 1. Virtual environment kya hai aur kyun chahiye?
 
**Problem:** Tumhare laptop par ek hi Python installed hai. Ab:
 
- Project A ko `Django 3.2` chahiye
- Project B ko `Django 5.0` chahiye
Dono ek saath global Python mein install nahi ho sakte. Ek install karoge to doosra toot jayega. Isko **dependency conflict** kehte hain.
 
**Solution:** Har project ka apna **alag, isolated folder** jisme uski apni packages ho. Wahi virtual environment (venv) hai.
 
```mermaid
%%{init: {'theme':'base','themeVariables':{'fontSize':'15px'}}}%%
flowchart TB
    subgraph BAD [Bina venv: sab ek global jagah]
        G[Global Python<br/>Django ek hi version]:::bad
        PA1[Project A<br/>chahiye Django 3.2]:::proj --> G
        PB1[Project B<br/>chahiye Django 5.0]:::proj --> G
        G --> X[CONFLICT]:::err
    end
    subgraph GOOD [venv ke saath: har project ka apna]
        PA2[Project A]:::proj --> VA[.venv A<br/>Django 3.2]:::ok
        PB2[Project B]:::proj --> VB[.venv B<br/>Django 5.0]:::ok
    end
 
    classDef bad fill:#FFA8A8,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
    classDef err fill:#FF6B6B,stroke:#8B0000,stroke-width:2px,color:#ffffff
    classDef proj fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef ok fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
```
 
**Venv ke fayde:**
 
| Fayda | Matlab |
|---|---|
| Isolation | Ek project ki packages doosre ko disturb nahi karti |
| Reproducibility | Doosra developer same versions install kar sakta hai |
| Clean system | Global Python mein kachra jama nahi hota |
| Easy delete | Project khatam? `.venv` folder delete, kaam khatam |
| No admin rights | Install ke liye sudo / admin ki zaroorat nahi |
 
> **Golden rule:** Har project = ek venv. Global Python mein `pip install` karne se bacho.
 
---
 
## 2. Andar se kaise kaam karta hai
 
Venv koi jadoo nahi hai, ye sirf ek **folder** hai. Banane par ye structure milta hai:
 
```
my_project/
└── .venv/
    ├── pyvenv.cfg          # config: kaunse Python se bana
    ├── Scripts/            # Windows (Linux/Mac par: bin/)
    │   ├── python.exe      # Python ki copy/link
    │   ├── pip.exe
    │   ├── activate        # bash ke liye
    │   ├── activate.bat    # cmd ke liye
    │   └── Activate.ps1    # PowerShell ke liye
    ├── Lib/                # Windows (Linux/Mac: lib/python3.x/)
    │   └── site-packages/  # <- tumhari installed packages YAHAN jaati hain
    └── Include/
```
 
**Activate karne par kya hota hai?**
 
Activation sirf 3 chhote kaam karta hai:
 
1. `PATH` ke **sabse aage** `.venv/Scripts` (ya `bin`) jod deta hai
2. `VIRTUAL_ENV` environment variable set karta hai
3. Terminal prompt mein `(.venv)` dikhata hai
Isliye ab `python` ya `pip` likhne par terminal **pehle venv wala** dhundhta hai.
 
```mermaid
%%{init: {'theme':'base'}}%%
sequenceDiagram
    autonumber
    participant U as Tum (Terminal)
    participant S as activate script
    participant P as PATH
    rect rgb(255, 236, 179)
    U->>S: .venv\Scripts\Activate.ps1
    S->>P: .venv\Scripts ko sabse aage jodo
    S->>U: Prompt: (.venv) PS C:\project>
    end
    rect rgb(187, 222, 251)
    U->>P: python script.py
    P-->>U: Pehle .venv wala python mila, wahi chala
    end
    rect rgb(255, 205, 210)
    U->>S: deactivate
    S->>P: PATH wapas purana
    end
```
 
**Important facts:**
 
- Activate karna **zaroori nahi** hai. Seedha `.venv\Scripts\python.exe script.py` bhi chala sakte ho. Scripts, cron jobs aur servers mein aksar yahi hota hai.
- Venv ka folder **relocatable nahi** hai. Folder move/rename kiya to scripts toot sakti hain. Naya bana lo, requirements se 1 minute mein ready.
- Venv ke andar Python ek chhota "launcher" hota hai, poora Python nahi. Base Python uninstall karoge to venv bhi bekaar.
**Check karo tum venv mein ho ya nahi:**
 
```python
import sys
print(sys.prefix)        # venv ka path (agar active)
print(sys.base_prefix)   # asli Python ka path
print(sys.prefix != sys.base_prefix)   # True => venv ke andar ho
```
 
Terminal se:
 
```powershell
# Windows
where python
# Linux / Mac
which python
```
Output mein pehla path `.venv` ka hona chahiye.
 
---
 
## 3. Pehla venv banao (step by step)
 
### Step 1: Project folder banao
 
```bash
mkdir my_project
cd my_project
```
 
### Step 2: Venv banao
 
```bash
python -m venv .venv
```
 
Windows par `python` kaam na kare to `py -m venv .venv` try karo. Linux/Mac par `python3 -m venv .venv`.
 
Yahan `.venv` folder ka naam hai. Industry convention `.venv` hai (VS Code ise apne aap pehchaan leta hai). `venv` ya `env` bhi chalta hai.
 
### Step 3: Activate karo
 
| Terminal | Command |
|---|---|
| **PowerShell** (Windows) | `.venv\Scripts\Activate.ps1` |
| **CMD** (Windows) | `.venv\Scripts\activate.bat` |
| **Git Bash** (Windows) | `source .venv/Scripts/activate` |
| **Linux / Mac** (bash/zsh) | `source .venv/bin/activate` |
 
Prompt aisa dikhega:
 
```
(.venv) PS C:\my_project>
```
 
### Step 4: Packages install karo
 
```bash
pip install requests
```
 
### Step 5: Verify karo
 
```bash
pip list          # sirf venv ki packages dikhengi
pip show requests # location dekho: .venv ke andar hoga
```
 
### Step 6: Kaam khatam, deactivate
 
```bash
deactivate
```
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart LR
    A[1. folder banao]:::s1 --> B["2. python -m venv .venv"]:::s2
    B --> C[3. activate]:::s3
    C --> D["4. pip install packages"]:::s4
    D --> E[5. code likho / run karo]:::s5
    E --> F[6. deactivate]:::s6
 
    classDef s1 fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef s2 fill:#63E6BE,stroke:#0CA678,stroke-width:2px,color:#053d2f
    classDef s3 fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef s4 fill:#FFA94D,stroke:#D9480F,stroke-width:2px,color:#2b1100
    classDef s5 fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
    classDef s6 fill:#FF8787,stroke:#C92A2A,stroke-width:2px,color:#3a0a0a
```
 
**Useful options:**
 
```bash
python -m venv .venv --upgrade-deps         # pip ko latest kar do
python -m venv .venv --system-site-packages # global packages bhi dikhao (rarely chahiye)
python -m venv .venv --clear                # purana venv saaf karke naya banao
```
 
---
 
## 4. pip aur requirements.txt
 
### 4.1 pip ke zaroori commands
 
```bash
pip install requests                  # latest version
pip install requests==2.31.0          # exact version
pip install "requests>=2.28,<3"       # range
pip install -U requests               # upgrade
pip uninstall requests                # hatao
pip list                              # sab installed
pip list --outdated                   # purani packages
pip show requests                     # detail + location
pip check                             # broken dependencies check
python -m pip install --upgrade pip   # pip khud upgrade
```
 
> **Tip:** `python -m pip` use karna safe hai, kyunki ye guarantee karta hai ki jis `python` ko tum chala rahe ho usi ka pip chale. Kabhi kabhi plain `pip` kisi aur Python ka nikal aata hai.
 
### 4.2 requirements.txt: project ki shopping list
 
**Banana:**
 
```bash
pip freeze > requirements.txt
```
 
**Install karna (naye machine / teammate par):**
 
```bash
pip install -r requirements.txt
```
 
**Example file:**
 
```
Django==5.0.4
requests==2.31.0
python-dotenv==1.0.1
```
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart LR
    A[Tumhara venv<br/>packages installed]:::a -- "pip freeze > requirements.txt" --> B[(requirements.txt)]:::file
    B -- git push --> C[GitHub]:::gh
    C -- git clone --> D[Teammate ka laptop]:::b
    B -. copy .-> D
    D -- "python -m venv .venv<br/>pip install -r requirements.txt" --> E[Same environment<br/>ready]:::ok
 
    classDef a fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef file fill:#FFD43B,stroke:#E67700,stroke-width:2px,color:#3a2a00
    classDef gh fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
    classDef b fill:#FFA94D,stroke:#D9480F,stroke-width:2px,color:#2b1100
    classDef ok fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
```
 
### 4.3 pip freeze ki ek dikkat (dhyan do)
 
`pip freeze` **saari** packages likhta hai, jisme tumhari packages ki dependencies (sub-packages) bhi hoti hain. Maan lo tumne sirf `Django` install kiya, freeze mein `asgiref`, `sqlparse` bhi aa jayenge.
 
Isse file lambi aur manage karna mushkil ho jata hai. Do achhe tareeke:
 
**Tareeka 1: do files rakho**
 
```
requirements.in      # sirf tumhari direct packages (Django, requests)
requirements.txt     # poora freeze (exact versions, sab sub-packages)
```
 
**Tareeka 2: dev aur prod alag**
 
```
requirements.txt        # production ke liye
requirements-dev.txt    # testing, linting (pytest, black)
```
 
`requirements-dev.txt` ke andar pehli line `-r requirements.txt` likh do, to dev install mein prod wali bhi aa jayengi:
 
```
-r requirements.txt
pytest==8.1.1
black==24.3.0
```
 
> Aage chal ke `uv` ya `poetry` jaise tools ye sab apne aap handle karte hain (section 9).
 
### 4.4 Version pinning: kyun zaroori hai
 
| Style | Example | Risk |
|---|---|---|
| Pinned | `requests==2.31.0` | Sabse safe, reproducible |
| Range | `requests>=2.28,<3` | Library ke liye theek |
| Unpinned | `requests` | Kal ko update se tumhara code toot sakta hai |
 
**Application** (Django app, bot, API) mein **pin karo**. **Library** (jo doosre install karenge) mein range rakho.
 
---
 
## 5. Git ke saath: .gitignore aur sharing
 
> **Sabse badi galti:** `.venv` folder ko Git mein push kar dena. Wo hazaaron files ka hota hai, machine-specific hai, aur kisi kaam ka nahi.
 
**`.gitignore` mein ye jodo:**
 
```
.venv/
venv/
env/
__pycache__/
*.pyc
.env
```
 
`.env` bhi ignore karo, kyunki usme API keys aur passwords hote hain.
 
**Agar galti se `.venv` already commit ho gaya:**
 
```bash
git rm -r --cached .venv
git commit -m "Remove .venv from tracking"
```
 
**Repo ke saath kya push karna hai:**
 
| Push karo | Mat karo |
|---|---|
| Source code | `.venv/` |
| `requirements.txt` | `__pycache__/` |
| `.gitignore` | `.env` (secrets) |
| `README.md` (setup steps) | |
 
**README mein setup steps zaroor likho:**
 
```markdown
## Setup
python -m venv .venv
.venv\Scripts\Activate.ps1      # Windows PowerShell
pip install -r requirements.txt
```
 
---
 
## 6. VS Code mein venv select karna
 
Terminal mein activate karna alag baat hai, **VS Code ka editor (IntelliSense, run button, debugger)** ko bhi bataana padta hai ki kaunsa Python use karna hai.
 
**Steps:**
 
1. `Ctrl + Shift + P` dabao
2. Type karo: `Python: Select Interpreter`
3. List mein se `.venv` wala choose karo (path mein `.venv\Scripts\python.exe` dikhega)
4. Neeche right corner mein selected interpreter dikhne lagega
**Aur ye bhi:**
 
- Naya terminal kholne par VS Code `.venv` ko **apne aap activate** kar deta hai (agar interpreter select hai).
- Agar `import requests` par yellow underline aaye, matlab VS Code ko galat interpreter select hai. Upar wale steps dobara karo.
**PowerShell mein activate nahi ho raha? Error aaye:** `running scripts is disabled on this system`
 
```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```
 
Ye sirf tumhare user ke liye hai, admin nahi chahiye. Phir dobara activate karo.
 
---
 
## 7. Real project workflows
 
### 7.1 Django
 
```bash
mkdir blog && cd blog
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install django
django-admin startproject config .
pip freeze > requirements.txt
python manage.py runserver
```
 
### 7.2 FastAPI
 
```bash
mkdir api && cd api
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install "fastapi[standard]"
pip freeze > requirements.txt
fastapi dev main.py
```
 
### 7.3 Flask
 
```bash
mkdir app && cd app
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install flask
pip freeze > requirements.txt
flask run
```
 
### 7.4 Existing project clone kiya (sabse common situation)
 
```bash
git clone <repo-url>
cd <repo>
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart TD
    Q{Project kaisa hai?}:::q -- Naya --> N1[venv banao]:::n --> N2[activate] --> N3[pip install] --> N4[pip freeze > requirements.txt]
    Q -- Clone kiya --> C1[venv banao]:::c --> C2[activate] --> C3["pip install -r requirements.txt"]
    N4 --> R([Coding shuru]):::done
    C3 --> R
 
    classDef q fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef n fill:#63E6BE,stroke:#0CA678,stroke-width:2px,color:#053d2f
    classDef c fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef done fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
```
 
---
 
## 8. Python version management
 
**Venv ek baat nahi karta:** wo **Python ka version** nahi badalta. Venv usi Python se banta hai jisse tumne bana-ne wali command chalayi. Python 3.9 se venv banaya to wo 3.9 ka hi rahega.
 
**Alag version chahiye to:**
 
**Windows (py launcher, sabse aasaan):**
 
```powershell
py -0                      # installed versions ki list
py -3.11 -m venv .venv     # 3.11 se venv banao
```
 
**Linux/Mac:**
 
```bash
python3.11 -m venv .venv
```
 
**Tools jo versions khud install karte hain:**
 
| Tool | Platform | Kaam |
|---|---|---|
| `pyenv` | Linux/Mac | Kai Python versions install + switch |
| `pyenv-win` | Windows | Same, Windows ke liye |
| `uv` | Sab | Python versions download + venv + packages, sab ek tool mein |
 
`uv` se example:
 
```bash
uv python install 3.12
uv venv --python 3.12
```
 
---
 
## 9. Alternatives: tools comparison
 
| Tool | Kya hai | Kab use karein |
|---|---|---|
| **venv** | Python ke saath built-in | Seekhne ke liye, simple projects. **Yahin se shuru karo.** |
| **virtualenv** | venv ka purana, tez version | Purane Python versions (2.7) ya extra features |
| **pipenv** | venv + pip ek tool mein, `Pipfile` | Kuch purane projects mein milta hai |
| **poetry** | Dependency + packaging + publishing | Library publish karni ho, bade projects |
| **uv** | Rust mein bana, bahut tez, all-in-one | Naye projects, speed chahiye (pip se kaafi tez) |
| **conda** | Python + non-Python (C libs) packages | Data science, ML, GPU libraries |
 
```mermaid
%%{init: {'theme':'base'}}%%
flowchart TD
    S{Kya kar rahe ho?}:::q
    S -- Seekh rahe ho / simple project --> V[venv + pip]:::v
    S -- Data science, ML, GPU --> CO[conda]:::co
    S -- Library publish karni hai --> PO[poetry]:::po
    S -- Speed chahiye, modern workflow --> UV[uv]:::uv
 
    classDef q fill:#FFD166,stroke:#B8860B,stroke-width:2px,color:#1a1a1a
    classDef v fill:#69DB7C,stroke:#2B8A3E,stroke-width:2px,color:#0a2e12
    classDef co fill:#74C0FC,stroke:#1971C2,stroke-width:2px,color:#0b2540
    classDef po fill:#B197FC,stroke:#6741D9,stroke-width:2px,color:#1e0f4d
    classDef uv fill:#FFA94D,stroke:#D9480F,stroke-width:2px,color:#2b1100
```
 
**uv ka quick workflow (aage seekhne layak):**
 
```bash
uv init my_app            # project + pyproject.toml
cd my_app
uv add requests           # venv apne aap banta hai, package install + record
uv run python main.py     # activate kiye bina chalao
uv sync                   # clone ke baad: sab install
```
 
**Conda ka quick workflow:**
 
```bash
conda create -n myenv python=3.11
conda activate myenv
conda install numpy
conda deactivate
conda env list
```
 
> **Mere hisaab se seedha rasta:** pehle `venv + pip` achhe se seekho, kyunki baaki sab tools usi concept par bane hain. Phir `uv` try karo.
 
---
 
## 10. Docker aur venv
 
Docker container khud ek isolated environment hota hai, to **kya container mein venv ki zaroorat hai?**
 
- **Aksar nahi.** Container mein ek hi app chalti hai, to seedha `pip install` theek hai.
- Kuch log phir bhi venv use karte hain taaki system Python ke packages se mix na ho, aur multi-stage builds mein sirf `.venv` copy kar sakein.
**Simple Dockerfile example:**
 
```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python", "main.py"]
```
 
Local development mein venv rakho, Docker image mein `requirements.txt` se install karo. Dono ek hi file use karte hain, isliye `requirements.txt` ka sahi hona zaroori hai.
 
---
 
## 11. Common errors aur fixes
 
| Error / Problem | Wajah | Fix |
|---|---|---|
| `running scripts is disabled on this system` | PowerShell execution policy | `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` |
| `python` / `pip` not recognized | Python PATH mein nahi | Reinstall karte waqt "Add to PATH" tick karo, ya `py` use karo |
| `ModuleNotFoundError: No module named 'x'` | Package us venv mein install nahi, ya venv activate nahi | Activate karo, `pip install x`. `where python` se check karo |
| Package install kiya, VS Code underline dikha raha | Galat interpreter selected | `Python: Select Interpreter` se `.venv` choose karo |
| `pip install` global mein ho gaya | Venv activate nahi tha | `deactivate` ya naya terminal, `.venv` activate karke dobara install |
| Folder rename/move ke baad venv kaam nahi karta | Venv paths absolute hote hain | Purana `.venv` delete, naya banao, `pip install -r requirements.txt` |
| `error: externally-managed-environment` (Linux/Mac) | OS apne Python mein pip se install rokta hai | Venv ke andar install karo (ye error venv ke bahar aata hai) |
| Venv banane par `ensurepip is not available` (Ubuntu) | `venv` package missing | `sudo apt install python3-venv` |
| Venv mein purana package version atka hua | Cache ya purani install | `pip install --force-reinstall <pkg>` ya naya venv |
| Activate kiya par `(.venv)` nahi dikha | Galat path ya shell | Path/terminal type check karo (table section 3) |
 
**Nuclear option (jab sab bigad jaye):**
 
```bash
deactivate
rmdir /s /q .venv           # Windows CMD
# Remove-Item -Recurse -Force .venv   (PowerShell)
# rm -rf .venv              (Linux / Mac)
python -m venv .venv
# activate karo
pip install -r requirements.txt
```
 
Venv disposable hai. Ye isiliye accha hai ki bigad jaye to 2 minute mein dobara ban jata hai.
 
---
 
## 12. Best practices
 
1. **Har project ka alag venv.** Kabhi share mat karo.
2. **Venv project folder ke andar `.venv` naam se rakho.** Tools ise auto-detect karte hain.
3. **Global Python mein `pip install` mat karo.** Sirf venv ke andar.
4. **`python -m pip` use karo**, plain `pip` nahi, taaki sahi Python ka pip chale.
5. **`.venv` ko `.gitignore` mein daalo**, `requirements.txt` ko commit karo.
6. **Application ke liye versions pin karo.**
7. **Naya package install karo to `requirements.txt` update karo** (bhoolna aam galti hai).
8. **Dev aur prod dependencies alag rakho** (`requirements-dev.txt`).
9. **Secrets `.env` mein rakho, `.env` ko gitignore karo.**
10. **README mein setup steps likho**, taaki teammate ya future tum khud 5 minute mein setup kar sako.
11. **Venv ko "disposable" samjho.** Problem aaye to delete aur recreate.
12. **Kabhi-kabhi `pip list --outdated` dekho** aur security updates lagao.
---
 
## 13. Cheat sheet
 
```text
BANANA          python -m venv .venv
ACTIVATE (PS)   .venv\Scripts\Activate.ps1
ACTIVATE (CMD)  .venv\Scripts\activate.bat
ACTIVATE (Bash) source .venv/Scripts/activate      (Windows Git Bash)
ACTIVATE (Mac)  source .venv/bin/activate
DEACTIVATE      deactivate
 
INSTALL         python -m pip install <pkg>
SAVE            pip freeze > requirements.txt
RESTORE         pip install -r requirements.txt
LIST            pip list
CHECK           where python   (Windows)  |  which python  (Linux/Mac)
DELETE          deactivate, phir .venv folder delete
```
 
```mermaid
%%{init: {'theme':'base'}}%%
mindmap
  root((Virtual<br/>Environment))
    Kyun
      dependency conflict bachao
      reproducible setup
      clean system
    Kaise
      python -m venv .venv
      activate
      pip install
    Share
      requirements.txt
      .gitignore
      README steps
    Tools
      venv
      uv
      poetry
      conda
    Dikkat
      PowerShell policy
      galat interpreter
      PATH issue
```
 
---
 
## 14. Practice tasks
 
Khud karo, phir hi aage badho:
 
1. **Basic:** `demo` folder banao, usme venv banao, activate karo, `requests` install karo. `where python` (ya `which python`) se verify karo ki path `.venv` ka hai.
2. **Isolation test:** Do alag folders mein do venvs banao. Ek mein `requests==2.25.0`, doosre mein `requests==2.31.0` install karo. Dono mein `pip show requests` chalake dekho versions alag hain.
3. **Share test:** `pip freeze > requirements.txt` banao. `.venv` delete karo. Naya venv banake `pip install -r requirements.txt` se wapas lao. Kya sab wahi aaya?
4. **Debug karo:** Venv **bina activate kiye** `pip install flask` chalao. Package kahan install hua? (`pip show flask` ka `Location` dekho.) Isse samjho ki activation kyun zaroori hai.
5. **Bina activate chalao:** `.venv\Scripts\python.exe -c "import sys; print(sys.prefix)"` chalao. Dekho ki activate kiye bina bhi venv ka Python chal gaya.
6. **Challenge:** Ek chhota FastAPI ya Flask project banao, `.gitignore` aur `README` setup steps ke saath, GitHub par push karo. Phir kisi doosre folder mein clone karke sirf README ke steps follow karke chalao. Agar chal gaya, to tumhara setup sahi hai.
 

